# Union-Find (Disjoint Set Union) — Complete Guide

```
The Core Idea: Friend Groups
Imagine you're tracking friend groups at a school. When two people become friends, their entire groups merge. You need to answer: "Are these two people in the same friend group?"
That's exactly what Union-Find solves.

The Data Structure
At its heart, Union-Find uses just one array: parent[]
Index:   0  1  2  3  4
parent: [0, 1, 2, 3, 4]   ← Initially everyone is their own boss
Each person points to their "boss" (parent). If you're your own boss, you're the root of a group.
```

```
Two Operations
1. find(x) → "Who is the ultimate boss of x?"
Keep following the chain upward until you reach the root.
parent = [0, 0, 1, 2, 3]

find(4):
  4 → parent[4]=3 → parent[3]=2 → parent[2]=1 → parent[1]=0 → parent[0]=0 ✓
  Root = 0
2. union(x, y) → "Merge the groups of x and y"
Find both roots. Make one root point to the other.
union(3, 7):
  find(3) = rootA
  find(7) = rootB
  parent[rootB] = rootA   ← rootB's group now joins rootA's group
To check if connected: find(x) == find(y)

The Naive Problem → Two Optimizations
❌ Without optimizations, the tree can become a linked list:
1 → 2 → 3 → 4 → 5 → 6 → 7 → 8
find(8) takes O(n) steps. Very slow!

✅ Optimization 1: Path Compression
After finding the root, make EVERYONE point directly to the root.
pythondef find(x):
    if parent[x] != x:
        parent[x] = find(parent[x])  # ← This one line does it!
    return parent[x]
```

**Before** `find(4)`:
```
4 → 3 → 2 → 1 → 0(root)
```

**After** `find(4)`:
```
4 → 0(root)
3 → 0(root)
2 → 0(root)
Next time anyone calls find(), it's instant!

✅ Optimization 2: Union by Rank
Always attach the SMALLER tree under the LARGER tree.
Without this, you could always attach large under small, creating tall trees again.
pythonrank = [0] * n   # rank = approximate height of tree

def union(x, y):
    px, py = find(x), find(y)
    if px == py: return   # Already same group!
    
    if rank[px] < rank[py]:
        px, py = py, px       # Make px the bigger one
    parent[py] = px           # Attach smaller under bigger
    if rank[px] == rank[py]:
        rank[px] += 1         # Only grows when equal

```

## Complete Template (Memorize This!)

In [3]:
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n
        self.components = n        # bonus: track number of groups

    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])  # path compression
        return self.parent[x]

    def union(self, x, y):
        px, py = self.find(x), self.find(y)
        if px == py:
            return False           # Already connected (cycle detected!)
        if self.rank[px] < self.rank[py]:
            px, py = py, px
        self.parent[py] = px
        if self.rank[px] == self.rank[py]:
            self.rank[px] += 1
        self.components -= 1       # Two groups merged into one
        return True

    def connected(self, x, y):
        return self.find(x) == self.find(y)


```

---

## Visual Example: Step by Step
```
nodes = [0, 1, 2, 3, 4]
edges = [(0,1), (1,2), (3,4), (0,3)]
```

**Start:**
```
parent = [0, 1, 2, 3, 4]    (5 separate groups)
```

**union(0,1):**
```
parent = [0, 0, 2, 3, 4]    
Group: {0,1}  {2}  {3}  {4}
```

**union(1,2):** → find(1)=0, find(2)=2
```
parent = [0, 0, 0, 3, 4]    
Group: {0,1,2}  {3}  {4}
```

**union(3,4):**
```
parent = [0, 0, 0, 3, 3]    
Group: {0,1,2}  {3,4}
```

**union(0,3):** → find(0)=0, find(3)=3
```
parent = [0, 0, 0, 0, 3]    
Group: {0,1,2,3,4}   ← Everything connected!

## Patterns in LeetCode Problems

### Pattern 1: Cycle Detection

"Does adding this edge create a cycle?"

Problems: Redundant Connection, Graph Valid Tree

In [4]:
# If find(u) == find(v) BEFORE union → cycle!
for u, v in edges:
    if not uf.union(u, v):
        return [u, v]   # This edge is redundant

SyntaxError: 'return' outside function (1893808661.py, line 4)

### Pattern 2: Count Connected Components

"How many separate islands/groups exist?"

Problems: Number of Connected Components, Number of Islands

In [ ]:
uf = UnionFind(n)
for u, v in edges:
    uf.union(u, v)
return uf.components   # Already tracked in our template!

### Pattern 3: Dynamic Connectivity

"Are these two nodes connected right now?"

Problems: Friend Circles, Accounts Merge

### Pattern 4: Grid Problems

Convert 2D grid to 1D index

Problems: Number of Islands (union-find version), Surrounded Regions

In [ ]:
# Convert (row, col) to single index
def index(r, c):
    return r * cols + c

# Then use normally
uf.union(index(r1,c1), index(r2,c2))

```
When to Use Union-Find?
Ask yourself these questions:

Am I grouping things together?
Do I need to check if two things are in the same group?
Are edges/connections being added (not removed)?
Do I need to detect cycles?

If yes to any → Union-Find is likely the right tool.
```

```
Complexity
OperationWith | both optimizations
find()          O(α(n)) ≈ O(1)
union()         O(α(n)) ≈ O(1)
Space           O(n)

α(n) = inverse Ackermann function. For any real input, it's ≤ 4. Essentially constant.
```

## The 3 Cases of Union

### Case 1: rank[px] < rank[py]

We attach the shorter tree under the taller tree.

py's height didn't change at all — px just got tucked under one branch. The overall height is still determined by py's longest path. So rank stays the same.

```
px (rank 1)        py (rank 3)
    A                  D
   / \                / \
  B   C              E   F
                        / \
                       G   H

After union(px, py) → attach px under py:

           D  (rank stays 3)
          /|\
         E F A
          /\ /\
         G H B C
```

### Case 2: rank[px] > rank[py]

Same logic, mirror image. The taller tree (px) doesn't get taller by absorbing a shorter one.

### Case 3: rank[px] == rank[py] ← only this case grows

Now A has a longer path than before (A→D→E or A→D→F = depth 2). The height genuinely increased, so rank must go up.

```
px (rank 1)      py (rank 1)
    A                D
   / \              / \
  B   C            E   F

After attaching py under px:

        A   (now rank 2 ✓)
       /|\
      B C D
         / \
        E   F
```

**Rank only grows when two equal-height trees merge, because that's the only time the resulting tree is taller than both originals.**

In [ ]:
class UnionFind:
    def __init__(self, n):
        self.parents = list(range(n))  # ultimate root of each node
        self.rank = [0] * n  # height of each tree
        self.components = n  # no. of disjoint sets
        
    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]
    
    def union(self, x, y):
        px, py = self.find(x), self.find(y)
        
        if px == py:  # two nodes are already connected/ have the same root (cycle detected) 
            return False  
        
        if self.rank[px] < self.rank[py]:
            px, py = py, px
        
        self.parent[py] = px  # always make the highest rank/ highest tree the root/parent of the smaller one
        
        if self.rank[px] == self.rank[py]:
            self.rank[px] += 1
        
        self.components -= 1
        return True  # merged both trees
    
    def connected(self, x, y):
        return self.find(x) == self.find(y)

In [ ]:
class UnionFind:
    def __init__(self, n):
        self.parents = list(range(n))
        
    def union(self, child, parent):
        self.parents[self.find(child)] = self.find(parent)
        
    def find(self, x):
        if x != self.parents[x]:
            self.parents[x] = self.find(self.parents[x])
        return self.parents[x]

## Leetcode Examples

### 547. Number of Provinces

https://leetcode.com/problems/number-of-provinces/?envType=problem-list-v2&envId=graph

In [ ]:
from collections import deque


class Solution:
    def findCircleNum(self, isConnected: List[List[int]]) -> int:
        def find(x):
            if roots[x] != x:
                roots[x] = find(roots[x])
            return roots[x]

        def union(x, y):
            nonlocal disj_sets
            px, py = find(x), find(y)
            if px == py:
                return False
            
            if ranks[px] < ranks[py]:
                px, py = py, px
            roots[py] = px

            if ranks[px] == ranks[py]:
                ranks[px] += 1

            disj_sets -= 1

            return True

        n = len(isConnected)
        roots = list(range(n))
        ranks = [0] * n
        disj_sets = n

        for c1 in range(n):
            for c2 in range(c1, n):
                if c1 != c2 and isConnected[c1][c2] == 1:
                    union(c1, c2)
        
        return disj_sets


    def findCircleNum2(self, isConnected: List[List[int]]) -> int:
        n = len(isConnected)
        visited = set()
        count = 0

        def dfs(city):
            for c in range(n):
                if isConnected[city][c] == 1 and c not in visited:
                    visited.add(c)
                    dfs(c)
        
        for city in range(n):
            if city not in visited:
                visited.add(city)
                dfs(city)
                count += 1
        
        return count
                

    def findCircleNum1(self, isConnected: List[List[int]]) -> int:
        n = len(isConnected)
        visited = set()
        np = 0
        
        for i in range(n):
            if i in visited:
                continue
            np += 1
            q = deque([i])
            visited.add(i)
            while q:
                city = q.popleft()
                row = isConnected[city]
                for c_city in range(n):
                    if row[c_city] == 1 and c_city != city and (c_city not in visited):
                        visited.add(c_city)
                        q.append(c_city)
        return np




        

### 684. Redundant Connection

https://leetcode.com/problems/redundant-connection/description/?envType=problem-list-v2&envId=graph

In [ ]:
from collections import defaultdict, deque

class Solution:
    def findRedundantConnection(self, edges: List[List[int]]) -> List[int]:
        n = len(edges)
        parents = list(range(n + 1))
        rank = [0] * (n + 1) 
        unique_trees = n + 1

        def find (x):
            if parents[x] != x:
                parents[x] = find(parents[x])
            return parents[x]

        def union(x, y):
            nonlocal unique_trees
            px, py = find(x), find(y)

            if px == py:  # same parent (cycle found)
                return False

            if rank[px] < rank[py]:
                px, py = py, px
            
            parents[py] = px  # always assign the larger tree as the root

            if rank[px] == rank[py]:
                rank[px] += 1
            unique_trees -= 1

            return True

        for (a, b) in edges:
            if not union(a, b):  # single union call will iterate from leaf to the root by calling find(x)                
                # print(parents)
                # print(rank)
                # print(unique_trees)
                return [a, b]



### 721. Accounts Merge

https://leetcode.com/problems/accounts-merge/

In [ ]:
from collections import defaultdict


class UF:
    def __init__(self, n):
        self.parents = list(range(n))
    
    def find(self, x):
        if self.parents[x] != x:
            self.parents[x] = self.find(self.parents[x])
        return self.parents[x]

    def union(self, child, parent):
        self.parents[self.find(child)] = self.find(parent)


class Solution:
    def accountsMerge(self, accounts: List[List[str]]) -> List[List[str]]:
        n = len(accounts)
        uf = UF(n)

        owners = {}
        for i, (_, *emails) in enumerate(accounts):
            for email in emails:
                if email in owners:
                    uf.union(i, owners[email])
                owners[email] = i

        res = defaultdict(list)
        for email, owner in owners.items():
            res[uf.find(owner)].append(email)
        
        return [[accounts[i][0]] + sorted(res[i]) for i in res]


### 1584. Min Cost to Connect All Points

https://leetcode.com/problems/min-cost-to-connect-all-points/

This question is bout finding the Minimum Spanning Tree (MST). 
There are two algorithms to solve this.
1. Prims Algo
2. Krushkal's Algo

#### Prims Algo

```
Prim's Algorithm is another method for finding the Minimum Spanning Tree. It starts from an arbitrary node and greedily chooses the edge with the smallest weight that connects a visited and an unvisited node.

The Mechanics of Prim's Algorithm in "Min Cost to Connect All Points"
Initialize Priority Queue:

Start from an arbitrary point and initialize a minimum priority queue with its edges.
Visited Nodes Tracking:

Keep track of visited nodes to ensure that each node is visited exactly once.
Iterate and Add to MST:

Pop the edge with the smallest weight from the priority queue. If the edge leads to an unvisited node, add the edge's weight to the total MST weight, and insert all edges from that node into the priority queue.
Completion Check:

Continue this process until all nodes have been visited.

Time and Space Complexity:
Time Complexity: O(n^2.logn), due to priority queue operations.
Space Complexity: O(n), for storing the priority queue and visited nodes.
```

#### Krushkal's Algo

```
Kruskal's Algorithm is an algorithm to find the Minimum Spanning Tree of a graph. It sorts all the edges by weight and adds them one by one, checking that the addition of each edge doesn't form a cycle.

The Essence of Kruskal's Algorithm in "Min Cost to Connect All Points"
Initialize Edge List:

Calculate the Manhattan distance between all pairs of points to form an edge list. Each edge is represented by a tuple (w, u, v), where w is the weight (Manhattan distance) and u and v are the nodes (points).
Sort the Edges:

Sort all edges by their weights. This helps us ensure that we're considering the smallest weight first, adhering to the "minimum" in Minimum Spanning Tree.
Union-Find for Connectivity:

Use a Union-Find data structure to keep track of connected components. This is crucial for efficiently checking whether adding a new edge would create a cycle.
Iterate and Add to MST:

Iterate through the sorted edge list, adding each edge to the Minimum Spanning Tree if it doesn't form a cycle. Keep a counter of the number of edges added, and stop when you've added n−1 edges, where n is the number of nodes.

Time and Space Complexity:
Time Complexity: O(n^2.logn), mainly due to sorting the edge list.
Space Complexity: O(n^2), for storing the edge list and Union-Find data structure.
```

In [ ]:
import heapq


class UnionFind:
    def __init__(self, n):
        self.parents = list(range(n))
        self.rank = [0] * n
        self.groups = n

    def find(self, x):
        if self.parents[x] != x:
            self.parents[x] = self.find(self.parents[x])
        return self.parents[x]

    def union(self, child, parent):
        pc, pp = self.find(child), self.find(parent)
        
        if pc == pp: # cycle
            return False

        if self.rank[pp] < self.rank[pc]:
            pc, pp = pp, pc

        self.parents[pc] = pp

        if self.rank[pc] == self.rank[pp]:
            self.rank[pp] += 1
        
        self.groups -= 1
        
        return True

def distance(pt1, pt2):
    return abs(pt1[0] - pt2[0]) + abs(pt1[1] - pt2[1])

"""
The quetsions indirectly asks us to find the MST of the connected graph
1. Prims Algorithm
- employs a Priority Queue to select edges with minimum weights iteratively
2. Krushkal's Algorithm
- utilizes a Union-Find data structure to find the MST efficiently.
"""

class Solution:
    def minCostConnectPoints1(self, points: List[List[int]]) -> int:
        """using kruskal's algo"""
        n = len(points)
        uf = UnionFind(n)
        distances = []

        for i in range(n):
            for j in range(n):
                d = distance(points[i], points[j])
                heapq.heappush(distances, (d, i, j))
        
        min_dist_sum = 0
        nodes_added = 0 
        while distances:
            w, u, v = heapq.heappop(distances)

            if uf.union(u, v): # if no cycle forms (so no need for visited node tracking)
                min_dist_sum += w
                nodes_added += 1

        return min_dist_sum


    def minCostConnectPoints(self, points: List[List[int]]) -> int:
        """prims algorithm"""
        n = len(points)
        dist_dict = {}
        heap = [(0, 0)]  # tart with a random node (weight, node_no)
        visited = set()
        tot_min_dist = 0

        while heap:
            w, u = heapq.heappop(heap)

            if u in visited or dist_dict.get(u, float("inf")) < w:  # this is the filter that only gets the min value for a node 
                continue

            visited.add(u)
            tot_min_dist += w

            for v in range(n):
                d = distance(points[u], points[v])
                if d < dist_dict.get(v, float("inf")):
                    dist_dict[v] = d
                    heappush(heap, (d, v)) # no need to worry as how may get pushed as it is a min heap the top holds the minimum distances
        return tot_min_dist    




### 1319. Number of Operations to Make Network Connected

https://leetcode.com/problems/number-of-operations-to-make-network-connected/?envType=problem-list-v2&envId=graph

In [ ]:
from collections import defaultdict


class UnionFind:
    def __init__(self, n):
        self.parents = list(range(n))
        self.n_groups = n

    def find(self, x):
        if self.parents[x] != x:
            self.parents[x] = self.find(self.parents[x])
        return self.parents[x]

    def union(self, x, y):
        px, py = self.find(x), self.find(y)
        
        if px == py:
            return False
        
        self.parents[py] = px
        self.n_groups -= 1

        return True

class Solution:
    def makeConnected(self, n: int, connections: List[List[int]]) -> int:
        n_conn = len(connections)
        if n_conn < n - 1: # at least to be a tree
            return -1

        uf = UnionFind(n)

        for con in connections:
            uf.union(con[0], con[1])
        
        print(uf.parents, uf.n_groups)

        return uf.n_groups - 1
        
